# Real Scan Pose Estimation Pipeline (Step-by-Step)

This notebook breaks the pose estimation process into simple, visual steps:
- **Step 1 (Before ICP)**: Visualizes the **Scanned Points (Blue)**, **YOLO Approximate Guess (Orange)**, and **OptiTrack Ground Truth (Green)**.
- **Step 2**: Runs ICP registration to align the YOLO guess to the scanned points.
- **Step 3 (After ICP)**: Visualizes the **Final Estimated Pose (Red)** vs **OptiTrack Ground Truth (Green)** and reports ADD accuracy in mm.

## 1. Helper Functions & Metrics

In [39]:
import os
import re
import time
import copy
import glob
import math
import random
import numpy as np
import open3d as o3d
from scipy.spatial.transform import Rotation as R
import pandas as pd
import matplotlib.pyplot as plt

def get_rotation_matrix_z(deg):
    """Creates a 3x3 rotation matrix for the Z-axis."""
    rad = math.radians(deg)
    c, s = math.cos(rad), math.sin(rad)
    return np.array([[c, -s, 0],
                     [s,  c, 0],
                     [0,  0, 1]])

def clean_and_crop_point_cloud(pcd, initial_pos, initial_angle, box_size, remove_plane=True, distance_threshold=3.0, plane_offset=0.5):
    """Removes ground/table plane and crops to Oriented Bounding Box."""
    if pcd.is_empty():
        return pcd
    pcd_clean = copy.deepcopy(pcd)
    if remove_plane:
        plane_model, inliers = pcd_clean.segment_plane(distance_threshold=distance_threshold, ransac_n=3, num_iterations=2000)
        [a, b, c, d] = plane_model
        pts = np.asarray(pcd_clean.points)
        distances = a * pts[:, 0] + b * pts[:, 1] + c * pts[:, 2] + d
        above_plane_indices = np.where(distances > plane_offset)[0]
        pcd_clean = pcd_clean.select_by_index(above_plane_indices)
        
    rot_matrix = get_rotation_matrix_z(-initial_angle)
    obb = o3d.geometry.OrientedBoundingBox(
        center=np.array(initial_pos),
        R=rot_matrix,
        extent=np.array(box_size)
    )
    return pcd_clean.crop(obb)

def merge_real_scans(data_dir, viewpoint_indices, yolo_pos, yolo_angle, crop_box, remove_plane=True):
    """Loads, cleans, and merges specified viewpoint point clouds."""
    merged_pcd = o3d.geometry.PointCloud()
    if viewpoint_indices is None or len(viewpoint_indices) == 0:
        pcd_files = [f for f in os.listdir(data_dir) if f.startswith('view') and f.endswith('.pcd') and '_surface' not in f and '_feature' not in f]
    else:
        pcd_files = []
        for idx in viewpoint_indices:
            candidate = f"view{idx}.pcd"
            if not os.path.exists(os.path.join(data_dir, candidate)):
                candidate = f"view{idx:02d}.pcd"
            pcd_files.append(candidate)
            
    for file_name in pcd_files:
        path = os.path.join(data_dir, file_name)
        if os.path.exists(path):
            raw = o3d.io.read_point_cloud(path)
            clean = clean_and_crop_point_cloud(raw, yolo_pos, yolo_angle, crop_box, remove_plane=remove_plane)
            merged_pcd += clean
            
    return merged_pcd

def preprocess_normal(pcd, num_points=False, invert_normals=False, radius=2, max_nn=30):
    """Estimates normals and computes FPFH feature descriptors."""
    if num_points and len(pcd.points) >= num_points:
        pcd_down = pcd.farthest_point_down_sample(num_points)
    else:
        pcd_down = copy.deepcopy(pcd)
    if len(pcd_down.points) == 0:
        return pcd_down, None
    avg_dist = np.mean(pcd_down.compute_nearest_neighbor_distance())
    pcd_down.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=avg_dist * radius, max_nn=max_nn))
    if invert_normals:
        normals = np.asarray(pcd_down.normals)
        for i in range(len(normals)):
            if normals[i][2] < 0:
                normals[i] *= -1
    fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down, o3d.geometry.KDTreeSearchParamHybrid(radius=avg_dist * 5, max_nn=100)
    )
    return pcd_down, fpfh

def calculate_add(source_cloud, T_est, T_gt):
    """
    Average Distance of Model Points (ADD) in millimeters:
    ADD = 1/|M| sum ||(T_est * x) - (T_gt * x)||_2
    """
    points = np.asarray(source_cloud.points)
    ones = np.ones((points.shape[0], 1))
    points_homo = np.hstack([points, ones])
    est_points = (T_est @ points_homo.T).T[:, :3]
    gt_points = (T_gt @ points_homo.T).T[:, :3]
    distances = np.linalg.norm(est_points - gt_points, axis=1)
    return np.mean(distances)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    o3d.utility.random.seed(seed)

set_seed(42)


## Step 1: Visualizing BEFORE ICP (Initial State)
This step loads everything and displays the initial spatial setup before registration:
- 🔵 **Blue Points**: Real Scanned Point Cloud (`view287.pcd`)
- 🟠 **Orange Model**: YOLO Approximate Pose (Where the registration starts)
- 🟢 **Green Model**: OptiTrack Ground Truth Pose (The true physical target)
- 🟩 **Green Box**: Oriented Cropping Bounding Box

In [ ]:
# ==========================================
# CONFIGURATION (EDIT THESE LINES)
# ==========================================
EXPERIMENT = "test_10_realgrasp"
WORKPIECE = "workpiece31"

VIEWPOINT_INDICES = [226]     # 1 view / ID(s) to test

# VIEWPOINT_INDICES = [226]
# VIEWPOINT_INDICES = [226, 137]
# VIEWPOINT_INDICES = [226, 137, 237]
# VIEWPOINT_INDICES = [226, 137, 237, 38]
# VIEWPOINT_INDICES = [226, 137, 237, 38, 239, 165, 273, 155]
# VIEWPOINT_INDICES = [226, 137, 237, 38, 239, 165, 273, 155, 180, 88, 188, 80]

# VIEWPOINT_INDICES = [80]     # 1 view / ID(s) to test
# VIEWPOINT_INDICES = [80, 88] # 2 view
# VIEWPOINT_INDICES = [80, 85, 91] # 3 view
# VIEWPOINT_INDICES = [80, 84, 88, 92] # 4 view 
# VIEWPOINT_INDICES = [80, 82, 84, 86, 88, 90, 92, 94] # 8 view
# VIEWPOINT_INDICES = [80, 81, 83, 84, 85, 87, 88, 89, 91, 92, 93, 95] # 12 view

# VIEWPOINT_INDICES = list(range(10))
CROP_BOX = [85, 105, 70]        # [X_width, Y_length, Z_height] in mm
NUMBER_OF_POINTS = 40000        # Sampling density for CAD model


# Translation and Rotation adjustment for CAD vs OptiTrack local axes:
# GT_OFFSET_MM = [-4, 4, 6] # optitrack aneh huhu
GT_OFFSET_MM = [0, 0, 0] # ya semoga aja average bisa
GT_ROTATION_Z_DEG = 90.0


# Paths
DATA_DIR = f"pcd_data/testing_data/{EXPERIMENT}"
if not os.path.exists(DATA_DIR):
    DATA_DIR = f"pcd_data/testing_data/{EXPERIMENT}/{WORKPIECE}"
CAD_PATH = f"workpiece/{WORKPIECE}/workpiece.stl"

# 1. Load OptiTrack Ground Truth Pose
# optitrack_path = os.path.join(DATA_DIR, "T_optitrack.npy")
optitrack_path = os.path.join(DATA_DIR, "T_average.npy")
if not os.path.exists(optitrack_path):
    eval_path = f"evaluation_result/{EXPERIMENT}/{WORKPIECE}/merge_full_transformation.npy"
    if os.path.exists(eval_path):
        optitrack_path = eval_path
    else:
        raise FileNotFoundError(f"OptiTrack file not found at {optitrack_path}!")
T_gt_raw = np.load(optitrack_path)
T_gt_raw[3, :] = [0, 0, 0, 1]  # Ensure valid 4x4 homogeneous matrix

# T_average.npy is already rotated to the workpiece frame, so GT_ROTATION_Z_DEG is 0.0
if 'T_average' in optitrack_path:
    GT_ROTATION_Z_DEG = 0.0

# Apply local Z-rotation adjustment
rad_z = math.radians(GT_ROTATION_Z_DEG)
T_local_rot = np.eye(4)
T_local_rot[:3, :3] = np.array([
    [math.cos(rad_z), -math.sin(rad_z), 0],
    [math.sin(rad_z),  math.cos(rad_z), 0],
    [0, 0, 1]
])
T_gt = T_gt_raw @ T_local_rot
T_gt[:3, 3] += GT_OFFSET_MM
T_gt[3, :] = [0, 0, 0, 1]

print(f"[Loaded] OptiTrack Ground Truth Pose (with {GT_ROTATION_Z_DEG} deg local rotation applied)")

# 2. Load YOLO Initial Guess
yolo_pose_path = os.path.join(DATA_DIR, "initial_obj_pose.npy")
if os.path.exists(yolo_pose_path):
    tf_obj = np.load(yolo_pose_path)
    YOLO_POS = tf_obj[:3].copy()
    YOLO_ANGLE = float(tf_obj[4])
    YOLO_POS[0] -= 0
    YOLO_POS[1] -= -10.0
else:
    YOLO_POS = [540.0, -70.0, 20.0]
    YOLO_ANGLE = 0.0

yolo_tf_path = os.path.join(DATA_DIR, "T_base2ob_yolo.npy")
if os.path.exists(yolo_tf_path):
    T_initial_guess = np.load(yolo_tf_path)
    T_initial_guess[3, :] = [0, 0, 0, 1]
else:
    T_initial_guess = np.eye(4)
    T_initial_guess[:3, :3] = get_rotation_matrix_z(-YOLO_ANGLE)
    T_initial_guess[:3, 3] = YOLO_POS

# 3. Load CAD Model & Real Scans
mesh = o3d.io.read_triangle_mesh(CAD_PATH)
mesh.compute_vertex_normals()
cad_cloud = mesh.sample_points_uniformly(number_of_points=NUMBER_OF_POINTS)

target_cloud = merge_real_scans(DATA_DIR, VIEWPOINT_INDICES, YOLO_POS, YOLO_ANGLE, CROP_BOX, remove_plane=True)

# 4. Calculate Initial Error (YOLO Guess vs Ground Truth)
initial_add = calculate_add(cad_cloud, T_initial_guess, T_gt)
print("="*60)
print("INITIAL STATE SUMMARY (BEFORE REGISTRATION)")
print("="*60)
print(f"OptiTrack Ground Truth Position (mm): {np.round(T_gt[:3, 3], 2)}")
print(f"YOLO Initial Guess Position (mm)    : {np.round(T_initial_guess[:3, 3], 2)}")
print(f"Initial ADD Error                   : {initial_add:.4f} mm")
print("="*60)

# 5. Create Geometries for Visualization
# Blue: Real Scanned Points
pcd_vis = copy.deepcopy(target_cloud)
pcd_vis.paint_uniform_color([0.0, 0.65, 0.93])

# Orange: YOLO Initial Guess CAD
cad_yolo = copy.deepcopy(cad_cloud).transform(T_initial_guess)
cad_yolo.paint_uniform_color([1.0, 0.5, 0.0])

# Green: OptiTrack Ground Truth CAD
cad_gt = copy.deepcopy(cad_cloud).transform(T_gt)
cad_gt.paint_uniform_color([0.0, 0.8, 0.2])

# Green Wireframe Box: Cropping Box
rot_matrix = get_rotation_matrix_z(-YOLO_ANGLE)
obb = o3d.geometry.OrientedBoundingBox(center=np.array(YOLO_POS), R=rot_matrix, extent=np.array(CROP_BOX))
obb.color = (0.0, 1.0, 0.0)

# Coordinate Frames
robot_base_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=100.0, origin=[0, 0, 0])
workpiece_gt_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=50.0).transform(T_gt)
yolo_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=35.0).transform(T_initial_guess)

print("\nOpening BEFORE-ICP 3D Visualization Window...")
print("  - 🔵 BLUE   : Real Scanned Points")
print("  - 🟠 ORANGE : YOLO Approximate Pose (Where ICP Starts)")
print("  - 🟢 GREEN  : OptiTrack Ground Truth (Target True Pose)")
print("  - 🟩 BOX    : Cropping Bounding Box (Green Wireframe)")
# o3d.visualization.draw_geometries([pcd_vis, cad_yolo, cad_gt, obb, robot_base_frame, workpiece_gt_frame, yolo_frame], window_name=f"BEFORE ICP: YOLO (Orange) vs OptiTrack GT (Green) vs Scan (Blue)")

pcd_vis.estimate_normals()
o3d.visualization.draw_geometries([pcd_vis, cad_gt])



[Loaded] OptiTrack Ground Truth Pose (with 0.0 deg local rotation applied)
INITIAL STATE SUMMARY (BEFORE REGISTRATION)
OptiTrack Ground Truth Position (mm): [508.69 -98.83  -2.45]
YOLO Initial Guess Position (mm)    : [ 508.75 -108.91   10.83]
Initial ADD Error                   : 16.4454 mm

Opening BEFORE-ICP 3D Visualization Window...
  - 🔵 BLUE   : Real Scanned Points
  - 🟠 ORANGE : YOLO Approximate Pose (Where ICP Starts)
  - 🟢 GREEN  : OptiTrack Ground Truth (Target True Pose)
  - 🟩 BOX    : Cropping Bounding Box (Green Wireframe)


In [41]:
world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=50.0, origin=[0, 0, 0])
    
o3d.visualization.draw_geometries([world_frame, pcd_vis, cad_gt])


# pcd_vis (blue) : actual scanned
# cad_yolo (orange) : cad model which yolo-approximated
# cad_gt (green) : cad optitrack



## Step 2: Run Pose Estimation (Registration & ICP)
This step takes the **YOLO Initial Guess (Orange)** and aligns it to the **Scanned Points (Blue)** using ICP.

In [42]:
# ==========================================
# STEP 2: COARSE-TO-FINE ICP REGISTRATION
# ==========================================
# Preprocess point clouds for registration
source_transformed = copy.deepcopy(cad_cloud).transform(T_initial_guess)
source_down, source_fpfh = preprocess_normal(source_transformed)
target_down, target_fpfh = preprocess_normal(target_cloud, invert_normals=True)

USE_RANSAC = False  # Set True if initial orientation is completely unknown

start_time = time.time()
if USE_RANSAC:
    print("Running RANSAC Global Alignment...")
    ransac_res = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source_down, target_down, source_fpfh, target_fpfh,
        mutual_filter=True, max_correspondence_distance=25.0,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
        ransac_n=3,
        checkers=[o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(25.0)],
        criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(4000, 0.99)
    )
    current_trans = ransac_res.transformation
else:
    current_trans = np.eye(4)

print("Running Coarse-to-Fine ICP Alignment...")
# Start with a large search radius (25mm) to bridge the initial ~20mm gap,
# then progressively tighten down to 2mm for sub-millimeter precision!
ICP_STAGES = [10, 5, 2]

for stage_idx, thr in enumerate(ICP_STAGES):
    icp_res = o3d.pipelines.registration.registration_icp(
        source_down, target_down, thr, current_trans,
        o3d.pipelines.registration.TransformationEstimationPointToPoint(),
        o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=300)
    )
    current_trans = icp_res.transformation
    temp_est = current_trans @ T_initial_guess
    temp_add = calculate_add(cad_cloud, temp_est, T_gt)
    print(f"  Stage {stage_idx+1} (Threshold: {thr:4.1f} mm) -> ADD Error: {temp_add:6.4f} mm | Fitness: {icp_res.fitness:.4f} | RMSE: {icp_res.inlier_rmse:.4f}")

process_time = time.time() - start_time

# Compute Final Transformation Matrix & Final Metrics
T_est = current_trans @ T_initial_guess
final_add = calculate_add(cad_cloud, T_est, T_gt)

print("="*60)
print("POSE ESTIMATION RESULTS (AFTER COARSE-TO-FINE ICP)")
print("="*60)
print(f"Initial ADD Error Before ICP : {initial_add:.4f} mm")
print(f"FINAL ADD ACCURACY AFTER ICP : {final_add:.4f} mm")
print(f"Accuracy Improvement        : {initial_add - final_add:.2f} mm")
print(f"Final ICP Fitness            : {icp_res.fitness:.4f}")
print(f"Final ICP RMSE               : {icp_res.inlier_rmse:.4f} mm")
print(f"Processing Time              : {process_time:.2f} seconds")
print("="*60)
  

Running Coarse-to-Fine ICP Alignment...
  Stage 1 (Threshold: 10.0 mm) -> ADD Error: 1.5073 mm | Fitness: 0.7405 | RMSE: 4.3164
  Stage 2 (Threshold:  5.0 mm) -> ADD Error: 3.0450 mm | Fitness: 0.5513 | RMSE: 2.4905
  Stage 3 (Threshold:  2.0 mm) -> ADD Error: 3.5911 mm | Fitness: 0.2929 | RMSE: 1.1039
POSE ESTIMATION RESULTS (AFTER COARSE-TO-FINE ICP)
Initial ADD Error Before ICP : 16.4454 mm
FINAL ADD ACCURACY AFTER ICP : 3.5911 mm
Accuracy Improvement        : 12.85 mm
Final ICP Fitness            : 0.2929
Final ICP RMSE               : 1.1039 mm
Processing Time              : 2.34 seconds


## Step 3: Visualizing AFTER ICP (Estimated vs Ground Truth)
Displays the final result overlay:
- 🔴 **Red Model**: Estimated Pose (Where ICP aligned the CAD)
- 🟢 **Green Model**: OptiTrack Ground Truth Pose
- 🔵 **Blue Points**: Real Scanned Point Cloud

In [ ]:
# cad_est = copy.deepcopy(cad_cloud).transform(T_est)
# cad_est.paint_uniform_color([1.0, 0.0, 0.0])       # Red: Estimated Pose

# cad_gt_vis = copy.deepcopy(cad_cloud).transform(T_gt)
# cad_gt_vis.paint_uniform_color([0.0, 1.0, 0.0])    # Green: OptiTrack Ground Truth

# pcd_vis = copy.deepcopy(target_cloud)
# pcd_vis.paint_uniform_color([0.0, 0.65, 0.93])     # Blue: Real Scanned Data

# world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=50.0, origin=T_gt[:3, 3])

# print("\nOpening AFTER-ICP 3D Visualization Window...")
# print(f"  - 🔴 RED   : Algorithm Estimated Pose (ADD Error: {final_add:.4f} mm)")
# print(f"  - 🟢 GREEN : OptiTrack Ground Truth Pose")
# print(f"  - 🔵 BLUE  : Real Scanned Data")
# o3d.visualization.draw_geometries([cad_est, cad_gt_vis, pcd_vis, world_frame], window_name=f"AFTER ICP Result (ADD Error: {final_add:.4f} mm)")


: 